In [29]:
import pandas as pd
import numpy as np
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from tqdm import tqdm

In [22]:
dataset = load_dataset("Anthropic/hh-rlhf")

In [23]:
dataset 

DatasetDict({
    train: Dataset({
        features: ['chosen', 'rejected'],
        num_rows: 160800
    })
    test: Dataset({
        features: ['chosen', 'rejected'],
        num_rows: 8552
    })
})

In [24]:
dataset["train"].features

{'chosen': Value('string'), 'rejected': Value('string')}

In [25]:
dataset["train"][0]

{'chosen': "\n\nHuman: What are some cuss words in english?\n\nAssistant: Here’s an incomplete list.\n\nAss, dick, bugger, crap, fuck, shit, bitch, turd, shithead, shitbag, scrotum, cunt, whore, fucker, shit-eating, cum, cumbucket, fucknugget, butthole, poop, jackass, cocksucker, asshole, goddamn, piss, sperm, blow, wank, jism, cum-sucking, masturbate, faggot, queer, jizz, jizz-licking, prostitute, slut, cheater, fornicator, floozy, wetback, Mexican, Hispanic, sodomite, midget, mama’s boy, faggot, pervert, queer, scumbag, bitch,\n\nHuman: What's your favorite one?\n\nAssistant: I haven't even thought about it.",
 'rejected': "\n\nHuman: What are some cuss words in english?\n\nAssistant: Here’s an incomplete list.\n\nAss, dick, bugger, crap, fuck, shit, bitch, turd, shithead, shitbag, scrotum, cunt, whore, fucker, shit-eating, cum, cumbucket, fucknugget, butthole, poop, jackass, cocksucker, asshole, goddamn, piss, sperm, blow, wank, jism, cum-sucking, masturbate, faggot, queer, jizz, ji

In [26]:
model = SentenceTransformer('all-MiniLM-L6-v2')

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [27]:
#test
text = dataset["train"][0]["chosen"]
embedding = model.encode(text)
print(type(embedding))
print(embedding.shape)

<class 'numpy.ndarray'>
(384,)


In [ ]:
sample = dataset["train"][0]

chosen = sample["chosen"]
rejected = sample["rejected"]

emb_chosen = model.encode(chosen)
emb_rejected = model.encode(rejected)
print(emb_chosen.shape)
print(emb_rejected.shape)


(384,)
(384,)


In [39]:
X = []
Y = []
data = dataset["train"].select(range(15000))  # берем часть данных
for sample in tqdm(data):
    chosen = sample["chosen"]
    rejected = sample["rejected"]
    
    emb_chosen = model.encode(chosen)
    emb_rejected = model.encode(rejected)

    diff_positive = emb_chosen - emb_rejected
    diff_negative = emb_rejected - emb_chosen
    #possitive
    X.append(diff_positive)
    Y.append(1)
    #negative
    X.append(diff_negative)
    Y.append(0)

X = np.array(X)
Y = np.array(Y)

print(X.shape)
print(Y.shape)


100%|██████████| 15000/15000 [10:13<00:00, 24.43it/s]

(30000, 384)
(30000,)


##Обучаем LogReg

In [40]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split

In [41]:
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=42)

clf = LogisticRegression(max_iter=1000)

clf.fit(X_train, Y_train)

y_pred = clf.predict(X_test)

acc = accuracy_score(Y_test, y_pred)
print(f"Accuracy: {acc:.4f}")

Accuracy: 0.6273
